# 03 Statistical Analysis & Validation

Use the engineered dataset to quantify returns, build confidence intervals, run a light hypothesis test, and sanity-check data quality before strategies rely on it.

In [1]:
import os
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import load_config
from src.utils.logger import get_logger
from src.data.data_manager import DataManager

config = load_config()
logger = get_logger('notebook.stats')
data_manager = DataManager(config=config)

Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
2025-11-18 18:30:15,387 | INFO | src.data.market_data | Market Data handler initialized | base_url: https://data.alpaca.markets


## Load engineered features

In [2]:
df_features = data_manager.load_data('engineered_features')
if df_features.empty:
    raise RuntimeError("Run 02_feature_engineering.ipynb to generate engineered features before proceeding.")

df_features = df_features.copy()
returns = df_features.get("return_1d")
if returns is None or returns.isna().all():
    returns = df_features['close'].pct_change()
returns = returns.dropna()
logger.info("Using %s return observations for statistical analysis.", len(returns))
display(df_features[['close']].tail())

2025-11-18 18:30:17,789 | INFO | src.data.data_manager | Loading dataset engineered_features from data/processed/engineered_features.parquet
2025-11-18 18:30:17,911 | INFO | notebook.stats | Using 99 return observations for statistical analysis.


,close
t,
2024-05-17 04:00:00+00:00,924.79
2024-05-20 04:00:00+00:00,947.80
2024-05-21 04:00:00+00:00,953.86
2024-05-22 04:00:00+00:00,949.50
2024-05-23 04:00:00+00:00,1037.99


## Summaries

In [3]:
returns.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

count    99.000000
mean      0.008307
std       0.032840
min      -0.100046
5%       -0.034813
25%      -0.010617
50%       0.005797
75%       0.024875
95%       0.050914
max       0.164009
Name: return_1d, dtype: float64

## Confidence interval + hypoteshis test

In [5]:
def confidence_interval(data, confidence=0.95):
    mean = data.mean()
    std = data.std(ddof=1)
    n = len(data)
    return stats.t.interval(confidence, n - 1, loc=mean, scale=std / np.sqrt(n))

def ttest_gt_zero(data):
    t_stat, p_value = stats.ttest_1samp(data, 0.0, alternative='greater')
    return t_stat, p_value

ci_low, ci_high = confidence_interval(returns)
print(f"95% Confidence Interval for Mean Return: [{ci_low:.6f}, {ci_high:.6f}]")

t_stat, p_value = ttest_gt_zero(returns)
print(f"T-test Statistic: {t_stat:.4f}, P-value: {p_value:.4f}")

95% Confidence Interval for Mean Return: [0.001757, 0.014857]
T-test Statistic: 2.5167, P-value: 0.0067


## Data validation

In [6]:
def validate(df: pd.DataFrame) -> dict:
    completeness = 1 - df.isna().sum().sum() / (len(df) * df.shape[1])
    positive_ratio = (returns > 0).mean()
    return {
        'rows': len(df),
        'completeness': float(completeness),
        'positive_return_ratio': float(positive_ratio)
    }

quality = validate(df_features)
quality

{'rows': 99, 'completeness': 1.0, 'positive_return_ratio': 0.6060606060606061}